# Phase 6: Retrieval Pipeline

Evaluate Dense, Sparse (BM25), and Hybrid (RRF) retrieval methods across 3 chunking strategies.
Uses `qa_pairs_filtered.parquet` for evaluation.

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Detect environment
def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    REPO_ROOT = Path('/content/rag-vn-finance')
    if not REPO_ROOT.exists():
        print("Đang tải mã nguồn và cài đặt thư viện lần đầu...")
        subprocess.run(['git', 'clone', 'https://github.com/thong7d/rag-vn-finance.git', str(REPO_ROOT)])
        
        req_path = REPO_ROOT / 'requirements.txt'
        if req_path.exists():
            os.system(f'pip install -r "{req_path}" -q')
            
        print("Cài đặt hoàn tất. Đang tự động khởi động lại Kernel để nạp thư viện lõi (Numpy/Torch)...")
        os.kill(os.getpid(), 9) # Tự động ngắt tiến trình để ép Colab khởi động lại RAM
    else:
        print("Mã nguồn đã tồn tại. Bỏ qua cài đặt...")
else:
    REPO_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())

print(f"Project root: {REPO_ROOT}")
assert REPO_ROOT.exists(), f"Project root not found: {REPO_ROOT}"

src_path = str(REPO_ROOT)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Load file .env từ Google Drive vào hệ thống Colab
if IN_COLAB:
    from dotenv import load_dotenv
    load_dotenv('/content/drive/MyDrive/rag-vn-finance/.env') 

print("\nColab setup complete.")

## 1. Environment Setup & Model Loading
Import necessary libraries, load project configurations from `config.yaml`, and initialize the SentenceTransformer model used for encoding user queries.

In [ ]:
import json
import pandas as pd
import faiss
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer

from src.utils import load_config, resolve_path, ensure_dir
from src.indexing import load_bm25_index
from src.retrieval import DenseRetriever, SparseRetriever, HybridRetriever, calculate_metrics

# Load config
config = load_config()

# Load Model
model_name = config['embedding']['model_name']
device = config['embedding'].get('device', 'auto')
model = SentenceTransformer(model_name, device=device)
print(f"Loaded {model_name} on {model.device}")

## 2. Load Evaluation Data
Load the synthetic QA pairs generated in Phase 5. We use the full `qa_pairs_filtered.parquet` dataset to evaluate our retrieval methods.

In [ ]:
qa_dir = resolve_path(config['synthetic_qa'], 'output_dir')
qa_path = os.path.join(qa_dir, 'qa_pairs_filtered.parquet')

if not os.path.exists(qa_path):
    raise FileNotFoundError(f"{qa_path} not found. Ensure Phase 5 is completed.")

df_qa = pd.read_parquet(qa_path)
print(f"Loaded {len(df_qa)} QA pairs for evaluation.")

## 3. Retrieval Evaluation Loop
Iterate through all 3 chunking strategies (`fixed_size`, `sentence_aware`, `article_level`). For each strategy:
1. Load the corresponding FAISS and BM25 indices.
2. Instantiate Dense, Sparse, and Hybrid retrievers.
3. Run all evaluation queries and compute `Precision@K`, `Recall@K`, `MRR`, and `NDCG@10`.
4. Aggregate the metrics.

In [ ]:
strategies = config['chunking']['strategies']
top_k_dense = config['retrieval']['top_k_dense']
top_k_sparse = config['retrieval']['top_k_sparse']
top_k_hybrid = config['retrieval']['top_k_hybrid']
rrf_k = config['retrieval']['rrf_k']

index_base_dir = resolve_path(config['indexing'], 'output_dir')
bm25_base_dir = resolve_path(config['indexing'], 'bm25_dir')

results = []

for strategy in strategies:
    print(f"\n{'='*40}\nEvaluating Strategy: {strategy}\n{'='*40}")
    
    # --- Load Dense Index ---
    faiss_path = os.path.join(index_base_dir, strategy, "index.faiss")
    chunk_ids_path = os.path.join(index_base_dir, strategy, "chunk_ids.json")
    
    if not os.path.exists(faiss_path):
        print(f"Missing Dense index for {strategy}. Skipping.")
        continue
        
    faiss_index = faiss.read_index(faiss_path)
    with open(chunk_ids_path, 'r', encoding='utf-8') as f:
        dense_chunk_ids = json.load(f)
        
    dense_retriever = DenseRetriever(faiss_index, dense_chunk_ids, model)
    
    # --- Load Sparse Index ---
    try:
        bm25_index, sparse_chunk_ids = load_bm25_index(bm25_base_dir, strategy)
    except FileNotFoundError:
        print(f"Missing Sparse index for {strategy}. Skipping.")
        continue
        
    sparse_retriever = SparseRetriever(bm25_index, sparse_chunk_ids)
    
    # --- Hybrid Retriever ---
    hybrid_retriever = HybridRetriever(dense_retriever, sparse_retriever, rrf_k=rrf_k)
    
    # --- Evaluate ---
    strategy_metrics = {'Dense': [], 'Sparse': [], 'Hybrid': []}
    
    # Progress bar over the evaluation dataset
    for _, row in tqdm(df_qa.iterrows(), total=len(df_qa), desc=f"Evaluating {strategy}"):
        query = row['question']
        ground_truth_doc_id = row['doc_id']
        
        # Dense
        dense_res = dense_retriever.retrieve(query, top_k=top_k_dense)
        dense_ids = [cid for cid, _ in dense_res]
        strategy_metrics['Dense'].append(calculate_metrics(dense_ids, ground_truth_doc_id, k=top_k_dense))
        
        # Sparse
        sparse_res = sparse_retriever.retrieve(query, top_k=top_k_sparse)
        sparse_ids = [cid for cid, _ in sparse_res]
        strategy_metrics['Sparse'].append(calculate_metrics(sparse_ids, ground_truth_doc_id, k=top_k_sparse))
        
        # Hybrid
        hybrid_res = hybrid_retriever.retrieve(query, top_k=top_k_hybrid)
        hybrid_ids = [cid for cid, _ in hybrid_res]
        strategy_metrics['Hybrid'].append(calculate_metrics(hybrid_ids, ground_truth_doc_id, k=top_k_hybrid))
        
    # --- Aggregate and Store Results ---
    for method, metrics_list in strategy_metrics.items():
        df_m = pd.DataFrame(metrics_list)
        avg_metrics = df_m.mean().to_dict()
        
        res_row = {
            'Strategy': strategy,
            'Method': method
        }
        res_row.update(avg_metrics)
        results.append(res_row)
        
        # Format metrics for printing
        metrics_str = ", ".join([f"{k}: {v:.4f}" for k, v in avg_metrics.items()])
        print(f"{method:10} -> {metrics_str}")


## 4. Save Benchmark Results
Export the final evaluation results to `evaluation/retrieval_benchmark.csv` for use in subsequent phases or reporting.

In [ ]:
if results:
    df_results = pd.DataFrame(results)
    eval_dir = resolve_path(config['evaluation'], 'output_dir')
    ensure_dir(eval_dir)
    
    out_path = os.path.join(eval_dir, "retrieval_benchmark.csv")
    df_results.to_csv(out_path, index=False)
    print(f"\nSaved retrieval benchmark results to {out_path}")
    display(df_results)
else:
    print("No results to save. Ensure indexes and QA data are present.")